# 05 — Batch pipeline: N books in parallel, low-storage mode

Processes the whole corpus (or a slice of it) with the production pipeline. The engine
comes from `configs/pipeline.toml`:

* **`scantailor_mrc`** (default since 2026-06-07, prototyped in notebook 06):
  ScanTailor cleanup (split, deskew, content detection, margins, uniform page size,
  illumination, dewarp) → optional DocRes → Tesseract hOCR → **MRC PDF** via
  `recode_pdf`. Needs the `scantailor-deviant-cli` binary (`~/.local/bin` build or the
  Docker image). Outputs are ~5× smaller than legacy (fad sample: 10.1 → 1.8 MB).
* **`legacy`**: OpenCV preprocess → img2pdf → OCRmyPDF.

Mechanics (both engines):

* **Books in parallel as separate processes** — OCR APIs aren't thread-safe, and
  processes also parallelize the OpenCV/ScanTailor work
* **Low-storage mode** — after each book, its downloaded TIFFs and intermediate pages
  are deleted; only the final **PDF + text sidecar** stay in `output/<faculty>/`
* **Disk guard** — a worker waits before downloading if free space drops below `MIN_FREE_GB`
* **Resumable** — books with an existing final PDF are skipped; every finished book is
  appended to `output/batch_report.jsonl`, so interrupting the kernel loses nothing

Peak disk ≈ `PARALLEL_BOOKS` × (TIFFs + split halves + cleaned pages ≈ 3× book size).
With the median 6.5 GB book expect ~80 GB peaks at 4 workers — tune
`MIN_FREE_GB`/`PARALLEL_BOOKS` to your disk.


In [1]:
%load_ext autoreload
%autoreload 2

import json
import os
import tomllib
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from evilflowers_books_digitalizer import LocalCache, load_settings
from evilflowers_books_digitalizer.batch import expected_pdf, free_gb, process_book

PARALLEL_BOOKS = 4
JOBS_PER_BOOK = max(1, (os.cpu_count() or 4) // PARALLEL_BOOKS)  # legacy engine only
MIN_FREE_GB = 40.0   # workers pause new downloads below this
FACULTIES = None     # e.g. ["fad", "fei"] — None = all
LIMIT = 20           # books per run — None = everything pending (~days!)

settings = load_settings()
cache = LocalCache(settings.cache_dir)
report_path = settings.output_dir / "batch_report.jsonl"

from evilflowers_books_digitalizer.pipeline.factory import DEFAULT_CONFIG_PATH
engine = tomllib.load(DEFAULT_CONFIG_PATH.open("rb")).get("pipeline", {}).get("engine", "legacy")
print(f"engine: {engine} | {PARALLEL_BOOKS} workers, "
      f"free disk: {free_gb(settings.output_dir.parent):.0f} GB")


engine: scantailor_mrc | 4 workers, free disk: 66 GB


## Work list

All non-empty books from the cached inventory (notebook 01), minus those already digitized. Smallest books first — early failures surface cheaply and disk pressure stays low at the start.

In [2]:
pending, done = [], 0
for key in FACULTIES or settings.sources:
    stats = cache.load_stats(key)
    if stats is None:
        raise RuntimeError("no cached inventory — run notebook 01 first")
    for book in stats.books:
        if book.n_pages == 0:
            continue
        if expected_pdf(settings.output_dir, key, book.book_id).exists():
            done += 1
        else:
            pending.append(book)

pending.sort(key=lambda book: book.total_bytes)
todo = pending[:LIMIT] if LIMIT else pending

print(f"done: {done} | pending: {len(pending)} "
      f"({sum(b.total_bytes for b in pending) / 1e9:,.0f} GB to download)")
print(f"this run: {len(todo)} books, {sum(b.total_bytes for b in todo) / 1e9:.1f} GB, "
      f"{sum(b.n_pages for b in todo)} frames")

done: 19 | pending: 861 (6,341 GB to download)
this run: 20 books, 42.9 GB, 1815 frames


## Run

Interrupting the kernel is safe: finished books are already on disk and in the report; in-flight books are re-done on the next run (downloads resume, their cache is cleaned either way).

In [3]:
rows = []
counters = {"ok": 0, "skipped": 0, "error": 0, "gb": 0.0}

with ProcessPoolExecutor(max_workers=PARALLEL_BOOKS) as pool, report_path.open("a") as report:
    futures = {
        pool.submit(
            process_book,
            book.source,
            book.book_id,
            jobs=JOBS_PER_BOOK,
            min_free_gb=MIN_FREE_GB,
        ): book
        for book in todo
    }
    with tqdm(total=len(futures), unit="book", desc="digitizing", smoothing=0.1) as bar:
        for future in as_completed(futures):
            book = futures[future]
            try:
                row = future.result()
            except Exception as exc:  # noqa: BLE001 — even a crashed worker is just a row
                row = {"source": book.source, "book_id": book.book_id,
                       "status": "error", "error": f"worker crash: {exc}"[:500]}
            rows.append(row)
            report.write(json.dumps(row, ensure_ascii=False) + "\n")
            report.flush()

            counters[row["status"]] += 1
            counters["gb"] += row.get("pdf_mb", 0) / 1e3
            bar.update(1)
            bar.set_postfix(ok=counters["ok"], skip=counters["skipped"],
                            err=counters["error"], out=f"{counters['gb']:.1f}GB",
                            disk=f"{free_gb(settings.output_dir.parent):.0f}GB")

            status = row["status"]
            extra = f"{row.get('pdf_mb', 0):.0f} MB, {row.get('language')}, {row.get('minutes', 0):.0f} min" \
                if status == "ok" else row.get("error", "")
            bar.write(f"[{status:>7}] {row['source']}/{row['book_id'][:50]} — {extra}")

digitizing:   0%|          | 0/20 [00:00<?, ?book/s]

INFO evilflowers_books_digitalizer.pipeline.base: pipeline start: mtf_CVI_OPACID_MTF_8022719218 (5 steps)
INFO evilflowers_books_digitalizer.pipeline.base: pipeline start: mtf_CVI_OPACID_MTF_8022714887 (5 steps)
INFO evilflowers_books_digitalizer.pipeline.base: pipeline start: fei_FEI_9788080707484 (5 steps)
INFO evilflowers_books_digitalizer.pipeline.base: pipeline start: mtf_CVI_OPACID_MTF_8022716863 (5 steps)
CVI_OPACID_MTF_8022714887:  28%|██▊       | 27/97 [11:44<30:26, 26.09s/page]
INFO evilflowers_books_digitalizer.pipeline.base: pipeline start: fad_CVI_OPACID_FA_9788087545225 (5 steps)


[  error] mtf/CVI_OPACID_MTF_8022714887 — RemoteProtocolError: Server disconnected without sending a response.


CVI_OPACID_MTF_8022719218:  38%|███▊      | 35/92 [12:13<19:53, 20.95s/page]
INFO evilflowers_books_digitalizer.pipeline.base: pipeline start: fad_CVI_OPACID_FA_Zasady_pravidla_UP_4_84 (5 steps)


[  error] mtf/CVI_OPACID_MTF_8022719218 — RemoteProtocolError: Server disconnected without sending a response.


...

[  error] fad/CVI_OPACID_FA_9788415223702 — RemoteProtocolError: Server disconnected without sending a response.


CVI_OPACID_FEI_9788082080974: 100%|██████████| 91/91 [33:03<00:00, 21.80s/page]
INFO evilflowers_books_digitalizer.pipeline.base: step download         done in 1985.2s (fei_CVI_OPACID_FEI_9788082080974)
CVI_OPACID_FEI_8022704466: 100%|██████████| 63/63 [26:06<00:00, 24.86s/page]
INFO evilflowers_books_digitalizer.pipeline.base: step download         done in 1568.2s (fei_CVI_OPACID_FEI_8022704466)
INFO evilflowers_books_digitalizer.pipeline.steps.scantailor: fei_CVI_OPACID_FEI_9788082080974: 91 frames -> 91 pages (0 spreads split, mixed)
INFO evilflowers_books_digitalizer.pipeline.base: step scantailor       done in  142.9s (fei_CVI_OPACID_FEI_9788082080974)
INFO evilflowers_books_digitalizer.language: detected language sk (p=1.00)
INFO evilflowers_books_digitalizer.pipeline.base: step language         done in    4.0s (fei_CVI_OPACID_FEI_9788082080974)
INFO evilflowers_books_digitalizer.pipeline.steps.scantailor: fei_CVI_OPACID_FEI_8022704466: 63 frames -> 63 pages (0 spreads split, mix

[     ok] fei/CVI_OPACID_FEI_9788082080974 — 5 MB, slk+eng, 40 min


INFO evilflowers_books_digitalizer.pipeline.steps.mrc: fei_CVI_OPACID_FEI_8022704466: MRC PDF 1.9 MB (63 pages, lang=slk+eng)
INFO evilflowers_books_digitalizer.pipeline.base: step mrc              done in  187.9s (fei_CVI_OPACID_FEI_8022704466)
INFO evilflowers_books_digitalizer.pipeline.base: step enrich           done in    0.0s (fei_CVI_OPACID_FEI_8022704466)
INFO evilflowers_books_digitalizer.pipeline.base: pipeline done: fei_CVI_OPACID_FEI_8022704466


[     ok] fei/CVI_OPACID_FEI_8022704466 — 2 MB, slk+eng, 32 min


## Report

Cumulative across all runs (the JSONL persists). Failed books: fix the cause (or just re-run — transient WebDAV errors are common) and they'll be retried since no PDF exists for them.

In [4]:
report_df = pd.read_json(report_path, lines=True)
# keep the latest attempt per book
report_df = report_df.drop_duplicates(subset=["source", "book_id"], keep="last")

ok = report_df[report_df.status == "ok"]
print(f"ok: {len(ok)} | skipped: {(report_df.status == 'skipped').sum()} "
      f"| errors: {(report_df.status == 'error').sum()}")
if len(ok):
    print(f"output: {ok.pdf_mb.sum() / 1e3:.2f} GB | "
          f"{ok.n_pages.sum():,.0f} pages | "
          f"avg {ok.minutes.mean():.1f} min/book | "
          f"languages: {ok.language.value_counts().to_dict()}")

errors = report_df[report_df.status == "error"]
errors[["source", "book_id", "error"]] if len(errors) else report_df.tail(10)

ok: 16 | skipped: 0 | errors: 18
output: 0.26 GB | 1,101 pages | avg 14.5 min/book | languages: {'slk+eng': 16}


,source,book_id,error
20,mtf,CVI_OPACID_MTF_8022714887,RemoteProtocolError: Server disconnected witho...
21,mtf,CVI_OPACID_MTF_8022719218,RemoteProtocolError: Server disconnected witho...
22,fad,CVI_OPACID_FA_9788087545225,RemoteProtocolError: Server disconnected witho...
23,fei,FEI_9788080707484,RemoteProtocolError: Server disconnected witho...
24,fad,CVI_OPACID_FA_TEORIA_OCHRANY_PAMIATOK,RemoteProtocolError: Server disconnected witho...
25,mtf,CVI_OPACID_MTF_8022716863,RemoteProtocolError: Server disconnected witho...
26,fad,CVI_OPACID_FA_Zasady_pravidla_UP_4_84,RemoteProtocolError: Server disconnected witho...
27,fei,CVI_OPACID_FEI_9788089656042,RemoteProtocolError: Server disconnected witho...
28,fad,CVI_OPACID_FA_9788087545249,RemoteProtocolError: Server disconnected witho...
29,mtf,CVI_OPACID_MTF_9788056201169,RemoteProtocolError: Server disconnected witho...


---
**Comparing engines on the same books:** the run skips books whose final PDF exists —
to redo a sample with a new engine/settings, move or delete their PDFs from
`output/<faculty>/` first (the JSONL report keeps one row per attempt, latest wins).

**Scaling up:** set `LIMIT = None` and let it run. For an unattended run, the
equivalent script is:

```python
from evilflowers_books_digitalizer.batch import process_book  # + the work-list code above
```

in a plain `python` process under `caffeinate`, or inside the Docker image (see
`Dockerfile` — all tooling baked in, mount `.cache/`, `output/` and `credentials.toml`).
